<a href="https://colab.research.google.com/github/danielomopariola95-cloud/BNP/blob/main/churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
b
# ============================================================================

# STEP 1: Install packages
!pip install pandas numpy scikit-learn plotly -q

# STEP 2: Import tools
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)
import warnings
warnings.filterwarnings("ignore")

print("="*60)
print("CUSTOMER CHURN PREDICTION SYSTEM")
print("="*60)

# STEP 3: Load or generate data
np.random.seed(42)
n = 1000

# Generate realistic customer data
df = pd.DataFrame({
    'gender': np.random.choice(['Male', 'Female'], n),
    'SeniorCitizen': np.random.choice([0, 1], n, p=[0.8, 0.2]),
    'Partner': np.random.choice(['Yes', 'No'], n),
    'Dependents': np.random.choice(['Yes', 'No'], n),
    'tenure': np.random.randint(0, 72, n),
    'PhoneService': np.random.choice(['Yes', 'No'], n),
    'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n),
    'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n),
    'PaperlessBilling': np.random.choice(['Yes', 'No'], n),
    'MonthlyCharges': np.random.uniform(20, 120, n),
    'TotalCharges': np.random.uniform(100, 8000, n),
    'Churn': np.random.choice(['Yes', 'No'], n, p=[0.26, 0.74])
})

print(f"Data loaded: {len(df)} customers")

# STEP 4: Clean and encode data
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df_encoded = pd.get_dummies(df, drop_first=True)

# STEP 5: Prepare features and target
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

# STEP 6: Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training data: {len(X_train)} customers")
print(f"Test data: {len(X_test)} customers")

# STEP 7: Train model
print("\nTraining Random Forest model...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# STEP 8: Make predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# STEP 9: Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# STEP 10: Print results
print("\n" + "="*60)
print("MODEL PERFORMANCE")
print("="*60)
print(f"Accuracy:  {accuracy:.2%}  (Overall correct guesses)")
print(f"Precision: {precision:.2%}  (When we say churn, are we right?)")
print(f"Recall:    {recall:.2%}  (What % of real churners did we catch?)")
print(f"F1 Score:  {f1:.2%}  (Balance between precision and recall)")
print(f"ROC AUC:   {roc_auc:.2f}  (Ability to separate churners from stayers)")
print("="*60)

# STEP 11: Feature importance
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTOP 10 FACTORS PREDICTING CHURN:")
print("-"*40)
for i, row in importance.head(10).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.2%}")

# STEP 12: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nCONFUSION MATRIX:")
print("-"*40)
print(f"True Negatives:  {cm[0][0]}  (Correctly predicted: No Churn)")
print(f"False Positives: {cm[0][1]}  (Wrongly predicted: Churn - false alarm)")
print(f"False Negatives: {cm[1][0]}  (Missed: Actually churned but predicted no)")
print(f"True Positives:  {cm[1][1]}  (Correctly predicted: Churn)")

# STEP 13: Create visualizations
print("\nGenerating visualizations...")

# Chart 1: Churn Distribution
fig1 = px.histogram(
    df,
    x="Churn",
    color="Churn",
    title="Customer Churn Distribution",
    labels={"Churn": "Customer Status", "count": "Number of Customers"},
    color_discrete_map={0: '#00bcd4', 1: '#ff6b6b'}
)

total = len(df)
churned = len(df[df['Churn'] == 1])
stayed = len(df[df['Churn'] == 0])

fig1.update_layout(template='plotly_dark', showlegend=False, bargap=0.2)
fig1.add_annotation(x=0, y=stayed, text=f"{stayed/total:.1%} stayed", showarrow=False)
fig1.add_annotation(x=1, y=churned, text=f"{churned/total:.1%} churned", showarrow=False)
fig1.show()

# Chart 2: Feature Importance
fig2 = px.bar(
    importance.head(10),
    x="Importance",
    y="Feature",
    orientation="h",
    title="Top 10 Factors Predicting Customer Churn",
    color="Importance",
    color_continuous_scale="Reds",
    text_auto='.1%'
)
fig2.update_layout(template='plotly_dark', height=400, xaxis_tickformat='.0%')
fig2.show()

# Chart 3: Confusion Matrix
fig3 = px.imshow(
    cm,
    text_auto=True,
    x=['Predicted No', 'Predicted Yes'],
    y=['Actual No', 'Actual Yes'],
    color_continuous_scale='Blues',
    title="Confusion Matrix"
)
fig3.update_layout(template='plotly_dark', height=400)
fig3.show()

print("\n" + "="*60)
print("Analysis Complete!")
print("="*60)